<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/Preliminary_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cosa possiamo testare e far vedere?**
## **prompting e generazione delle risposte:**
- Tasso di risposta corretta con le diverse strategie di similarità accoppiate con un modello Llama che risponde generando una risposta generica (non direttamente l'opzione corretta)
- Tasso di risposta corretta con il modello che genera direttamente l'opzione che gli diamo
- Dalla consegna: What prompt (zero, few shot, etc.) gives the best results?
- Dalla consegna: How sensitive are different LLMs to different prompts?
- Dalla consegna: What types of questions do the models tend to struggle on?
- Dalla consegna: Is the model often overconfident in its answers and does that affect its performance?
- Dalla consegna: Should the same prompt be used for all questions or made more specific as the questions get harder?
## **models:**
- Dalla consegna: Are some LLMs better than others at answering the questions?
- Dalla consegna: Are bigger models better than smaller models?
- Dalla consegna: Are the models performing as well as a human on this task?
- Dalla consegna: Are certain models better at certain topics than others? (Es: controllare se ci sono info sui dataset sui quali sono stati allenati)
- Dalla consegna: Are “thinking” models better at answering questions than “non-thinking” ones?
## **improvements:**
- Dalla consegna: Is there a way to fine-tune a model to improve its performance? If so, what data could you use to train the model?

In [1]:
# Clear widget metadata before saving/downloading
import IPython
IPython.display.clear_output()

In [2]:
class Model():
  def __init__(self, name, model, answer_function):
    self.name = name
    self.model = model
    self.answer_function = answer_function

  def answer(self, question):
    return 0


In [3]:
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 83.2 MB/s eta 0:00:00


In [4]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [5]:
from huggingface_hub import login
login(HF_TOKEN)

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [7]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [9]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

In [10]:
messages = [
    {"role": "system", "content": "Sei un esperto di storia."}, # Rispondi Ad una delle opzioni (A, B, C o D)
    {"role": "user", "content": "Chi è napoleone? A. Un politico, B: un personaggio televisivo C. STO CAZZO D. Un calciatore"},
]

In [11]:
generation_args = {
    "max_new_tokens": 600,
    "return_full_text": False,
    "temperature": 0.5,
    "do_sample": True,
}

In [14]:
output = pipe(messages, **generation_args)
answer = output[0]["generated_text"]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [18]:
answer_letter, answer_full = answer.split("\n", 1)
print(answer_letter)
print("="*80)
print(answer_full)

La risposta corretta è A: un politico.

Napoleone Bonaparte (1769-1821) fu un politico, un generale e un condottiero francese che divenne il primo re dell'Unione francese e dell'Impero francese. Fu un leader militare e politico di grande fama, che si distinse per la sua capacità di conquistare e unificare gran parte dell'Europa. La sua vita e le sue azioni hanno avuto un impatto profondo sulla storia dell'Europa e del mondo.

Napoleone è stato amato e rispettato da molti per la sua visione di un'Europa unita e prospera, ma anche per la sua capacità di conquistare e di creare un impero. Tuttavia, la sua vita è stata anche caratterizzata da una serie di battaglie e di sconfitte, che lo hanno costretto a lasciare l'Europa e a cercare un nuovo luogo per rifugiarsi.

La figura di Napoleone è stata oggetto di numerosi film, libri e serie televisive, che hanno cercato di rappresentare la sua vita e le sue azioni in modo fedele e credibile.


## Answer Extraction: Matching Model Output to Options

The LLM generates free-form text, but the game expects a single choice: **A, B, C, or D**.
We implement three strategies (in order of increasing sophistication) to map the model response back to one of the provided options:

1. **Regex extraction** – look for explicit letter mentions (fast & zero-cost)
2. **TF-IDF + Cosine Similarity** – Vector Space Model approach covered in the lectures
3. **Sentence-BERT (sBERT) Semantic Similarity** – deep embedding approach from the semantic search lectures

A combiner function tries each strategy in order, falling back to the next one if no confident match is found.

In [19]:
# Strategy 1: Regex-based direct extraction
import re

def extract_by_regex(model_output: str):
    text = model_output.strip()

    m = re.search(r'(?:answer(?:\s+is)?|option|choice|select|pick|correct)\s*[:\-]?\s*\*{0,2}([ABCD])\*{0,2}', text, re.IGNORECASE)
    if m: return m.group(1).upper()

    m = re.search(r'\b([ABCD])[\)\.:](?:\s|$)', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # lone capital letter (models often deliberate, then conclude with the letter)
    letters = re.findall(r'(?<![a-zA-Z])([ABCD])(?![a-zA-Z])', text)
    if letters:
        return letters[-1].upper()
    return None

test_cases = [
    ('The answer is A, because Napoleon was a political figure.', 'A'),
    ('Napoleon was a ruler. I think the answer is B.', 'B'),
    ('C) Un politico', 'C'),
    ('Napoleon era un generale. La risposta corretta e D.', 'D'),
    ('He was born in Corsica and rose to become emperor', None),
]
print('Regex extraction tests:')
for text, expected in test_cases:
    result = extract_by_regex(text)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'  [{status}] Input: {repr(text[:55]):57s} -> Got: {result}, Expected: {expected}')

Regex extraction tests:
  [PASS] Input: 'The answer is A, because Napoleon was a political figur' -> Got: A, Expected: A
  [PASS] Input: 'Napoleon was a ruler. I think the answer is B.'          -> Got: B, Expected: B
  [PASS] Input: 'C) Un politico'                                          -> Got: C, Expected: C
  [PASS] Input: 'Napoleon era un generale. La risposta corretta e D.'     -> Got: D, Expected: D
  [PASS] Input: 'He was born in Corsica and rose to become emperor'       -> Got: None, Expected: None


In [20]:
!pip install -q scikit-learn

In [21]:
# Strategy 2: TF-IDF + Cosine Similarity (Vector Space Model)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def pick_by_tfidf(model_output: str, options: dict):

    labels = list(options.keys())
    texts = [model_output] + [options[l] for l in labels]

    # Fit TF-IDF on all texts together so the vocabulary is shared
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)


    query_vec = tfidf_matrix[0]
    option_vecs = tfidf_matrix[1:]

    sims = cosine_similarity(query_vec, option_vecs)[0]
    scores = {labels[i]: float(sims[i]) for i in range(len(labels))}
    best = max(scores, key=scores.get)
    return best, scores

options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}
model_response = 'Napoleon era un grande leader politico e militare, imperatore dei francesi.'
best_tfidf, scores_tfidf = pick_by_tfidf(model_response, options_test)
print('TF-IDF cosine similarity scores:')
for label, score in sorted(scores_tfidf.items(), key=lambda x: -x[1]):
    marker = ' <-- SELECTED' if label == best_tfidf else ''
    print(f'  {label} ({options_test[label]!r:30s}): {score:.4f}{marker}')
print(f'\nPicked option: {best_tfidf}')

TF-IDF cosine similarity scores:
  A ('Un politico'                 ): 0.3286 <-- SELECTED
  D ('Un calciatore'               ): 0.0923
  B ('un personaggio televisivo'   ): 0.0696
  C ('qualcosa di assurdo'         ): 0.0000

Picked option: A


In [22]:
!pip install -q sentence-transformers

In [23]:
# Strategy 3: Sentence-BERT (sBERT) Semantic Similarity

from sentence_transformers import SentenceTransformer
import numpy as np

sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def pick_by_sbert(model_output: str, options: dict):

    labels = list(options.keys())
    all_texts = [model_output] + [options[l] for l in labels]

    embeddings = sbert_model.encode(all_texts, convert_to_numpy=True, normalize_embeddings=True)

    query_emb = embeddings[0]
    option_embs = embeddings[1:]

    # Dot product of unit vectors = cosine similarity
    sims = option_embs @ query_emb
    scores = {labels[i]: float(sims[i]) for i in range(len(labels))}
    best = max(scores, key=scores.get)
    return best, scores

best_sbert, scores_sbert = pick_by_sbert(model_response, options_test)
print('sBERT semantic similarity scores:')
for label, score in sorted(scores_sbert.items(), key=lambda x: -x[1]):
    marker = ' <-- SELECTED' if label == best_sbert else ''
    print(f'  {label} ({options_test[label]!r:30s}): {score:.4f}{marker}')
print(f'\nPicked option: {best_sbert}')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sBERT semantic similarity scores:
  A ('Un politico'                 ): 0.3535 <-- SELECTED
  D ('Un calciatore'               ): 0.1437
  B ('un personaggio televisivo'   ): 0.0086
  C ('qualcosa di assurdo'         ): -0.1167

Picked option: A


In [25]:
# Combined Answer Selector -- the full pipeline

def select_option(model_output: str, options: dict, use_sbert: bool = True, verbose: bool = False) -> str:
    # --- Stage 1: Regex ---
    regex_pick = extract_by_regex(model_output)
    if regex_pick and regex_pick in options:
        if verbose:
            print(f'[Strategy 1 - Regex] Confident match found: {regex_pick}')
        return regex_pick
    if verbose:
        print(f'[Strategy 1 - Regex] No match (got {regex_pick!r}), falling back to similarity...')

    # --- Stage 2/3: Similarity ---
    if use_sbert:
        best, scores = pick_by_sbert(model_output, options)
        method = 'Strategy 3 - sBERT'
    else:
        best, scores = pick_by_tfidf(model_output, options)
        method = 'Strategy 2 - TF-IDF'

    if verbose:
        sorted_scores = sorted(scores.items(), key=lambda x: -x[1])
        print(f'[{method}] Similarity scores:')
        for label, score in sorted_scores:
            marker = ' <-- SELECTED' if label == best else ''
            print(f'  {label} ({options[label]!r:35s}): {score:.4f}{marker}')
        gap = sorted_scores[0][1] - sorted_scores[1][1] if len(sorted_scores) > 1 else 1.0
        confidence = 'HIGH' if gap > 0.10 else ('MEDIUM' if gap > 0.03 else 'LOW')
        print(f'  Confidence gap: {gap:.4f} ({confidence})')

    return best

example_options = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'STO CAZZO',
    'D': 'Un calciatore'
}

raw_answer = answer_full
print('=' * 65)
print('Model raw output:')
print(raw_answer)
print('=' * 65)
final = select_option(raw_answer, example_options, use_sbert=True, verbose=True)
print('=' * 65)
print(f'FINAL ANSWER SUBMITTED TO GAME: {final}')

Model raw output:

Napoleone Bonaparte (1769-1821) fu un politico, un generale e un condottiero francese che divenne il primo re dell'Unione francese e dell'Impero francese. Fu un leader militare e politico di grande fama, che si distinse per la sua capacità di conquistare e unificare gran parte dell'Europa. La sua vita e le sue azioni hanno avuto un impatto profondo sulla storia dell'Europa e del mondo.

Napoleone è stato amato e rispettato da molti per la sua visione di un'Europa unita e prospera, ma anche per la sua capacità di conquistare e di creare un impero. Tuttavia, la sua vita è stata anche caratterizzata da una serie di battaglie e di sconfitte, che lo hanno costretto a lasciare l'Europa e a cercare un nuovo luogo per rifugiarsi.

La figura di Napoleone è stata oggetto di numerosi film, libri e serie televisive, che hanno cercato di rappresentare la sua vita e le sue azioni in modo fedele e credibile.
[Strategy 1 - Regex] No match (got None), falling back to similarity...
[S

## Summary: Three Strategies Compared

| Strategy | Method | Pros | Cons |
|---|---|---|---|
| **1. Regex** | Pattern matching | Zero cost, instant | Fails when model reasons without naming a letter |
| **2. TF-IDF + Cosine** | Sparse vector similarity (VSM lectures) | No extra model needed | Purely lexical — misses synonyms and paraphrases |
| **3. sBERT + Cosine** | Dense embedding similarity (semantic search lectures) | Captures meaning; cross-lingual | Needs a second model loaded in memory |

**Recommended default:** `select_option(..., use_sbert=True)`.

sBERT captures that *"leader politico"* ≈ *"Un politico"* even with no shared tokens — exactly the  
kind of semantic matching needed when the LLM explains its reasoning in full sentences rather than  
naming a letter explicitly. The multilingual model also handles Italian questions gracefully.

# **An Answering model with the performances**

Ridefinizione della classe Model (Claude -> sembra corretta e buona)

In [26]:
from typing import Callable

class Model():
    """
    Il modello generates the output.
    answer_fn decide how to get the final option.
    """

    def __init__(self, name: str, answer_fn: Callable[[str, dict], str]):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        pass

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        """Generates and process the answer through answer_fn."""
        raw_output = self.generate(question, system_prompt)
        return self.answer_fn(raw_output, options)

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"

class HFPipelineModel(Model):
    DEFAULT_GEN_ARGS = {
        "max_new_tokens": 600,
        "return_full_text": False,
        "temperature": 0.5,
        "do_sample": True,
    }

    def __init__(
        self,
        name: str,
        model_name: str,
        answer_fn: Callable[[str, dict], str],
        hf_token: str | None = None,
        device_map: str = "cuda",
        gen_args: dict | None = None,
        cache_dir: str | None = None,
    ):
        super().__init__(name, answer_fn)
        self.gen_args = {**self.DEFAULT_GEN_ARGS, **(gen_args or {})}

        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=True, token=hf_token, cache_dir=cache_dir,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        self._pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

    def generate(self, question: str, system_prompt: str = "Sei un assistente utile.") -> str:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ]
        output = self._pipe(messages, **self.gen_args)
        return output[0]["generated_text"]

## **Gestione della similarità**

In [27]:
def select_option_new(
    model_output: str,
    options: dict,
    use_regex: bool = True,
    use_sbert: bool = True,
    verbose: bool = False,
) -> str | None:
    """
    Estrae la risposta del modello confrontandola con le opzioni disponibili.

    Strategie applicate in cascata (ognuna fa fallback alla successiva):
      1. Regex      — cerca direttamente la lettera/chiave nell'output (se use_regex=True)
      2. TF-IDF     — similarità lessicale                            (se use_sbert=False)
      3. sBERT      — similarità semantica                            (se use_sbert=True)

    Args:
        model_output:  testo grezzo generato dal modello
        options:       dict tipo {"A": "testo opzione A", "B": "testo opzione B", ...}
        use_regex:     se True, prova prima a estrarre la risposta con regex
        use_sbert:     se True usa sBERT come fallback, altrimenti TF-IDF
        verbose:       se True stampa i dettagli di ogni stage

    Returns:
        La chiave dell'opzione scelta (es. "A"), oppure None se use_regex=True
        e non viene trovata nessuna corrispondenza (senza fallback di similarità).
    """

    # ── Stage 1: Regex ────────────────────────────────────────────────────────
    if use_regex:
        regex_pick = extract_by_regex(model_output)
        if regex_pick and regex_pick in options:
            if verbose:
                print(f'[Stage 1 - Regex] Match trovato: {regex_pick!r}')
            return regex_pick
        if verbose:
            print(f'[Stage 1 - Regex] Nessun match (got {regex_pick!r}), fallback a similarità...')

        # Se regex è abilitata ma non trova nulla e non c'è fallback → None
        if not use_sbert and not use_regex:
            return None

    # ── Stage 2/3: Similarity ─────────────────────────────────────────────────
    if use_sbert:
        best, scores = pick_by_sbert(model_output, options)
        method = 'Stage 3 - sBERT'
    else:
        best, scores = pick_by_tfidf(model_output, options)
        method = 'Stage 2 - TF-IDF'

    if verbose:
        sorted_scores = sorted(scores.items(), key=lambda x: -x[1])
        print(f'[{method}] Similarity scores:')
        for label, score in sorted_scores:
            marker = ' <-- SELECTED' if label == best else ''
            print(f'  {label} ({options[label]!r:35s}): {score:.4f}{marker}')
        gap = sorted_scores[0][1] - sorted_scores[1][1] if len(sorted_scores) > 1 else 1.0
        confidence = 'HIGH' if gap > 0.10 else ('MEDIUM' if gap > 0.03 else 'LOW')
        print(f'  Confidence gap: {gap:.4f} ({confidence})')

    return best

## **Define the model**

Per avere tutti i modelli separati e farli rispondere creo dizionario con tutto quello che serve ad ogni modello.

In [28]:
models = {}

In [29]:
# 1. Regex -> FallBack with TF-IDF
llama_regex_tfidf = HFPipelineModel(
    name="llama-regex-tfidf",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    answer_fn=lambda raw_output, opts: select_option(raw_output, opts, use_regex=True, use_sbert=False),
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
)

# 2. Only sBERT (senza regex)
llama_sbert = HFPipelineModel(
    name="llama-regex-sbert",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    answer_fn=lambda raw_output, opts: select_option(raw_output, opts, use_regex=False, use_sbert=True),
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
)

# 3. Only TF-IDF
llama_tf_idf = HFPipelineModel(
    name="llama-sbert",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    answer_fn=lambda raw_output, opts: select_option(raw_output, opts, use_regex=False, use_sbert=False),
    hf_token=HF_TOKEN,
    cache_dir="./models_cache",
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [30]:
models = {m.name: m for m in [llama_regex_tfidf, llama_tf_idf, llama_sbert]}

# Game Integration: Playing PoliMillionaire with our Models

In this section we wire our `HFPipelineModel` instances to the `MillionaireClient` API.

The key challenge is the **data translation layer** between the two systems:
- The API gives us `question.options` as a list of `Option` objects with `.id` (int) and `.text` (str)
- Our `select_option()` expects a `dict` mapping letter labels `{'A': ..., 'B': ...}` to option texts
- After selection, we need to map our letter label back to the `option.id` integer the API expects

We also need to handle the **30-second timeout** carefully: the LLM inference must complete before the deadline.

In [ ]:
import os

repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

In [ ]:
import sys
sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError

In [ ]:

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

In [ ]:
# Data translation

LABELS = ['A', 'B', 'C', 'D']

def options_to_dict(api_options) -> dict:
    """
    Convert the API's list of Option objects into the dict format
    expected by select_option().

    Example:
        [Option(id=0, text='Rome'), Option(id=1, text='Paris'), ...]
        --> {'A': 'Rome', 'B': 'Paris', ...}
    """
    return {LABELS[i]: opt.text for i, opt in enumerate(api_options)}

def label_to_option_id(label: str, api_options) -> int:
    """
    Convert a letter label ('A', 'B', ...) back to the integer option ID
    that game.answer() expects.

    Example:
        label='B', options=[Option(id=0,...), Option(id=1,...), ...]
        --> 1
    """
    idx = LABELS.index(label)
    return api_options[idx].id

def format_question_prompt(question_text: str, options_dict: dict) -> str:
    """
    Build the prompt string sent to the LLM, including the question
    and all labelled options.
    """
    options_str = '\n'.join(f'{label}: {text}' for label, text in options_dict.items())
    return f'{question_text}\n{options_str}'

# Quick test
from millionaire_client import Option
dummy_opts = [Option(id=0, text='Rome'), Option(id=1, text='Paris'), Option(id=2, text='Berlin'), Option(id=3, text='Madrid')]
d = options_to_dict(dummy_opts)
print('options_to_dict:', d)
print('label_to_option_id B ->', label_to_option_id('B', dummy_opts))
print('prompt:\n' + format_question_prompt('What is the capital of France?', d))

In [ ]:
# Core game loop: play one full game with a given model
import time
from millionaire_client import GameError

SYSTEM_PROMPT = (
    'You are a quiz expert. You will be given a multiple choice question with options A, B, C, D. '
    'Think step by step, then clearly state which option letter you choose as your final answer.'
)

def play_with_model(
    model,
    competition_id: int,
    client: MillionaireClient,
    verbose: bool = True,
) -> dict:
    """
    Play one complete game session using the given HFPipelineModel.

    The model generates a free-text answer; select_option_new() maps it
    to a letter label; label_to_option_id() converts to the API integer.

    Args:
        model          : an HFPipelineModel instance
        competition_id : which competition to play (0=Entertainment, 1=History, ...)
        client         : authenticated MillionaireClient
        verbose        : print question/answer details

    Returns:
        Summary dict with per-question logs and final stats.
    """
    game = client.game.start(competition_id=competition_id)
    if verbose:
        print(f'=== Game started | Session {game.session_id} | Model: {model.name} ===')

    log = []
    while game.in_progress:
        question = game.current_question
        if not question:
            break

        options_dict = options_to_dict(question.options)
        prompt       = format_question_prompt(question.text, options_dict)

        if verbose:
            print(f'\n--- Level {game.current_level} | Time left: {game.time_remaining:.1f}s ---')
            print(f'Q: {question.text}')
            for label, text in options_dict.items():
                print(f'  {label}: {text}')

        # --- LLM inference (timed) ---
        t0 = time.time()
        raw_output = model.generate(prompt, system_prompt=SYSTEM_PROMPT)
        inference_time = time.time() - t0

        # --- Map output to option ---
        chosen_label = select_option_new(
            raw_output, options_dict,
            use_regex=True, use_sbert=True, verbose=verbose
        )

        # Fallback: if select_option_new returns None, pick A to avoid timeout
        if chosen_label is None:
            chosen_label = 'A'
            if verbose:
                print('[Warning] No option selected, defaulting to A')

        chosen_id = label_to_option_id(chosen_label, question.options)

        if verbose:
            print(f'Model raw output: {raw_output[:120]!r}')
            print(f'Selected: {chosen_label} (id={chosen_id}) | Inference: {inference_time:.1f}s')


        result = game.answer(chosen_id)

        # Log the outcome
        entry = {
            'level'          : game.current_level,
            'question'       : question.text,
            'options'        : options_dict,
            'chosen_label'   : chosen_label,
            'chosen_text'    : options_dict[chosen_label],
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            'raw_output'     : raw_output,
        }
        log.append(entry)

        if verbose:
            status = 'CORRECT' if result.correct else ('TIMED OUT' if result.timed_out else 'WRONG')
            print(f'Result: {status} | Earned so far: ${result.earned_amount:,.2f}')

        if result.game_over:
            break

    summary = {
        'model'           : model.name,
        'competition_id'  : competition_id,
        'session_id'      : game.session_id,
        'final_level'     : game.current_level,
        'earned_amount'   : game.earned_amount,
        'num_questions'   : len(log),
        'num_correct'     : sum(1 for e in log if e['correct']),
        'num_timed_out'   : sum(1 for e in log if e['timed_out']),
        'avg_inference_s' : round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'             : log,
    }

    if verbose:
        print(f'\n=== Game Over ===')
        print(f'Reached level: {summary["final_level"]} | '
              f'Correct: {summary["num_correct"]}/{summary["num_questions"]} | '
              f'Earned: ${summary["earned_amount"]:,.2f} | '
              f'Avg inference: {summary["avg_inference_s"]}s')

    return summary

In [ ]:
# competition ID:
#   0 = Entertainment
#   1 = Ancient History and Politics
#   2 = Science and Nature
#   3 = Maths

COMPETITION_ID = 1

# Pick model to run
chosen_model = llama_regex_tfidf

result_summary = play_with_model(
    model=chosen_model,
    competition_id=COMPETITION_ID,
    client=client,
    verbose=True,
)

In [ ]:
# Benchmark: compare all three models across all competitions

COMPETITION_IDS = [0, 1, 2, 3]
all_results = []

for comp_id in COMPETITION_IDS:
    comp_name = client.competitions.list_all()[comp_id].name
    print(f'\n{"="*60}')
    print(f'Competition {comp_id}: {comp_name}')
    print(f'{"="*60}')

    for model_name, model in models.items():
        print(f'\n>>> Running model: {model_name}')
        summary = play_with_model(model, comp_id, client, verbose=False)
        all_results.append({
            'competition'    : comp_name,
            'model'          : model_name,
            'correct'        : summary['num_correct'],
            'total'          : summary['num_questions'],
            'accuracy'       : round(summary['num_correct'] / max(summary['num_questions'], 1), 3),
            'timed_out'      : summary['num_timed_out'],
            'earned'         : summary['earned_amount'],
            'avg_inference_s': summary['avg_inference_s'],
        })
        r = all_results[-1]
        print(f'  Accuracy: {r["correct"]}/{r["total"]} ({r["accuracy"]*100:.0f}%) | '
              f'Earned: ${r["earned"]:,.2f} | Timeouts: {r["timed_out"]} | '
              f'Avg inference: {r["avg_inference_s"]}s')

In [ ]:
# Results table and leaderboard check
import pandas as pd

# --- Results table ---
df = pd.DataFrame(all_results)
print('=== Benchmark Results ===')
print(df.to_string(index=False))

print('\n=== Best accuracy per competition ===')
print(df.groupby('competition')[['model', 'accuracy']].apply(lambda g: g.loc[g['accuracy'].idxmax()]))

print('\n=== Average accuracy per model (across all competitions) ===')
print(df.groupby('model')['accuracy'].mean().sort_values(ascending=False))

# --- Check leaderboard position ---
print('\n=== Current Leaderboard Positions ===')
for comp_id in COMPETITION_IDS:
    lb = client.leaderboard.get(competition_id=comp_id, limit=20)
    my_entry = client.leaderboard.find_player(competition_id=comp_id, username=USERNAME)
    rank_str = f'not ranked yet'
    if my_entry:
        rank = next((i+1 for i, e in enumerate(lb.entries) if e.username.lower() == USERNAME.lower()), '?')
        rank_str = f'rank #{rank} | ${my_entry.score:,.2f} | level {my_entry.reached_level}'
    print(f'  {lb.competition.name}: {rank_str}')